# Unit 4 exercises: hold the speed, follow the line

The starter cruises at 19.49 m/s with a target of 20 and steers at a waypoint without ever measuring how far it is from the line.
These exercises build the two controllers that fix that, one term at a time, and measure each one the way the lesson did: by its step response on a straight, and by races with noise.
Every number you produce is a number about racekit's **kinematic car**: a model, not the truth (Unit 8 re-identifies the car before any gain is trusted on CARLA).

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Functions you write in one exercise are reused in later ones, so run the exercises in order.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit.cars import KinematicCar
from racekit.measure import noise_aware_better, repeat_races, spread
from racekit.runner import CONTROL_DT, run_race
from racekit.tracks import load_track
from racekit.variants import write_variant

AGENTS = Path("../../projects/racekit/agents")
UNIT_04 = AGENTS / "unit-04.py"
DT = CONTROL_DT  # 0.05 s: one control tick; advance the car with 10 substeps per tick, as the runner does

# racekit's kinematic car (a model, not the truth)
WHEELBASE = 2.875  # m, L
DELTA_MAX = math.radians(70.0)  # wheel angle at |steer| = 1; delta = -steer * DELTA_MAX (a positive steer turns right)
A_THROTTLE = 6.0  # m/s^2 at full throttle
C_DRAG = 4.0e-4  # 1/m: drag deceleration C_DRAG * v * |v|


def wrap_angle(angle):
    # Unit 2: wrap an angle in radians to [-pi, pi)
    return (angle + math.pi) % (2.0 * math.pi) - math.pi

## Exercise 1 - The P loop's error, predicted and measured

A speed loop with only a P term sets `throttle = clip(kp * (target - v), 0, 1)` and `brake = clip(-kp * (target - v), 0, 1)`.

1. Write `equilibrium_error(kp, target)`: the speed error at which a P loop settles on a straight, from the drag balance A_THROTTLE·kp·e = C_DRAG·(target − e)² (the smaller root of the quadratic).
2. Write `p_loop_final_error(kp, target, seconds)`: drive `KinematicCar()` (jitter 0) straight from rest with that P loop, calling `car.advance(throttle, 0.0, brake, 0.0, 0, DT, 10)` once per tick, and return `target - car.v` at the end.
3. Evaluate both for the four cases (kp, target) = (0.05, 20), (0.2, 20), (1.0, 20) and (0.05, 30), with `seconds = 120`.
4. Find the P gain whose equilibrium error at a 20 m/s target is exactly 0.01 m/s, in closed form.

Define:

- `equilibrium_error`, `p_loop_final_error`: the functions;
- `CASES`: the list of the four (kp, target) pairs, in the order above;
- `predicted`, `simulated`: dicts from each (kp, target) pair to the predicted and the simulated error (m/s);
- `kp_for_one_cm`: the gain for a 0.01 m/s error at 20 m/s;
- `why_never_zero`: one sentence on why no P gain can make the error zero, in terms of the drag balance.

In [ ]:
def equilibrium_error(kp, target):
    return None  # TODO


def p_loop_final_error(kp, target, seconds):
    return None  # TODO


CASES = []  # TODO
predicted = {}  # TODO
simulated = {}  # TODO
kp_for_one_cm = None  # TODO
why_never_zero = "TODO"

## Exercise 2 - Windup, and conditional integration

Write the lesson's controller from its description, without copying it:

- `PID(kp, ki=0.0, kd=0.0, limits=None)` with `reset()` (zero the integral, forget the previous error) and `update(error, dt, rate=None)`, which returns `kp * e + ki * integral + kd * rate` (unclipped);
- with `rate=None` the D term uses the finite difference `(error - previous error) / dt` (0 on the first call);
- with `limits=(low, high)`, **conditional integration**: compute the candidate integral (the old one plus `error * dt`) and the P + I command with it (the D term is not part of this test); keep the candidate only if that command is strictly inside (low, high), or if it is at or above `high` with a negative error, or at or below `low` with a positive error; otherwise the integral does not change;
- expose the integral as the attribute `integral`.

Then write `step_response(controller, v0, target, seconds)` (drive `KinematicCar()` straight from `v0`, the positive part of the command as the throttle and the negative part as the brake, each clipped to [0, 1]; return a NumPy array of the speed after every tick, and a NumPy array of `controller.integral` after every tick) and `step_metrics(speeds, v0, target)`, which returns (overshoot as a percentage of the step, settling time in s, `target - speeds[-1]`), with the lesson's definitions: the settling time is the time of the first tick from which the speed stays within 2% of the step of the target for the rest of the run (the tick after the last one outside the band; the time of tick n, counting from 0, is (n + 1)·DT), `math.inf` if the last tick is outside, and 0 if no tick is.

The case: PI with (kp, ki) = (0.5, 0.1), from **10 m/s to 30 m/s**, for 40 s, once without limits and once with `limits=(0.0, 1.0)`.

Define:

- `PID`, `step_response`, `step_metrics`: the class and the functions;
- `metrics_plain`, `metrics_conditional`: the two (overshoot, settling, error) triples;
- `peak_integral_plain`, `peak_integral_conditional`: the largest value the integral reaches in each run (m);
- `fig`, `ax`: one plot of both speed traces against time, with the target as a horizontal line, axis labels, a title and a legend;
- `what_windup_is`: one or two sentences: what the integral integrated in the plain run that the controller could do nothing about, and how that produced the overshoot.

In [ ]:
class PID:
    def __init__(self, kp, ki=0.0, kd=0.0, limits=None):
        pass  # TODO

    def reset(self):
        pass  # TODO

    def update(self, error, dt, rate=None):
        return None  # TODO


def step_response(controller, v0, target, seconds):
    return None, None  # TODO: (speeds, integrals)


def step_metrics(speeds, v0, target):
    return None  # TODO: (overshoot %, settling s, error at the end)


metrics_plain = None  # TODO
metrics_conditional = None  # TODO
peak_integral_plain = None  # TODO
peak_integral_conditional = None  # TODO
fig, ax = None, None  # TODO
what_windup_is = "TODO"

## Exercise 3 - Tune by step response

A step from rest saturates the throttle for seconds; a small step saturates it for a few ticks at most.
Measure both, for a grid of gains, all with conditional integration on (0, 1).

- Gains: kp in (0.25, 0.5, 1.0, 2.0) and ki in (0.05, 0.1, 0.2, 0.4), 16 pairs.
- Steps: from 0 to 20 m/s (the big step) and from 18 to 20 m/s (the small step), 30 s each.
- The rule: among the pairs whose overshoot is at most 1% and whose |error at the end| is at most 0.002 m/s on **both** steps, choose the one with the smallest **larger** of its two settling times; break ties by the smaller kp, then the smaller ki.

Define:

- `grid`: a dict from each (kp, ki) pair to `(big, small)`, the two `step_metrics` triples;
- `best`: the chosen (kp, ki) pair;
- `best_settling`: its larger settling time (s);
- `big_step_range`: (the smallest, the largest) settling time of the big step over the 16 pairs (s);
- `saturation_note`: one or two sentences: for the pairs with kp ≥ 0.5 the big step's settling time hardly depends on the gains; why? And what makes the kp = 0.25 rows slower?

In [ ]:
KPS = (0.25, 0.5, 1.0, 2.0)
KIS = (0.05, 0.1, 0.2, 0.4)
grid = {}  # TODO
best = None  # TODO
best_settling = None  # TODO
big_step_range = None  # TODO
saturation_note = "TODO"

## Exercise 4 - The heading error is the D term

A straight line through the origin at heading θ = 30° (π/6 rad).
For a car at (x, y) with heading ψ, the cross-track error (left positive) is e = −sin θ·x + cos θ·y and the heading error is ψ_e = wrap(θ − ψ).

1. Show (on paper, then in a sentence) that ė = −v·sin ψ_e.
2. Simulate: start the car at the point 1.5 m to the left of the origin, (−1.5 sin θ, 1.5 cos θ), heading θ, at v = 20 m/s.
   Each tick, apply the drag-balancing throttle C_DRAG·v²/A_THROTTLE (with the car's current v), no brake, and the wheel angle δ = −K_E·e + K_HEAD·ψ_e with K_E = 0.2, converted to `steer = clip(-delta / DELTA_MAX, -1, 1)`; run 15 s.
   Do it for K_HEAD in (0.0, 0.25, 0.5, 1.0, 2.0).
   For each, record e at the start of every tick (before the advance), and report the **overshoot** (how far the car crosses to the other side: max(0, −min e)) and the **settling time**: the time of the first tick from which |e| ≤ 0.05 m for the rest of the run (computed exactly like Exercise 2's, with tick n stamped (n + 1)·DT even though e is recorded at its start; `math.inf` if the last tick is outside).
3. Check the derivative: for K_HEAD = 1.0, with e_n, v_n and ψ_e,n recorded at the start of tick n, compute the largest |(e_{n+1} − e_{n−1})/(2·DT) + v_n·sin ψ_e,n| over the ticks that have both neighbours (a central difference).
   Expect the largest gap on the first ticks, where δ changes fastest and the central difference is least accurate; look at how fast it shrinks.

Define:

- `why_e_dot`: the sentence for step 1;
- `lateral`: a dict from each K_HEAD to its (overshoot, settling) pair (m, s);
- `smallest_damped_k_head`: the smallest K_HEAD in the list with overshoot ≤ 0.05 m and a finite settling time;
- `rate_gap`: the largest gap of step 3 (m/s);
- `k_d_at_20`, `k_d_at_30`: the effective D gain K_D = K_HEAD / v for K_HEAD = 1.0 at 20 and at 30 m/s (s).

In [ ]:
THETA = math.pi / 6.0
K_HEADS = (0.0, 0.25, 0.5, 1.0, 2.0)
why_e_dot = "TODO"
lateral = {}  # TODO
smallest_damped_k_head = None  # TODO
rate_gap = None  # TODO
k_d_at_20 = None  # TODO
k_d_at_30 = None  # TODO

## Exercise 5 - Hold a circle, with and without the feedforward

The line is a circle of radius R centred at (0, R): from the origin, heading +x, it turns **left**.
For a car at (x, y) with heading ψ:

- the cross-track error (left positive: inside the turn is left) is e = R − distance from the centre;
- the line's heading at the car is θ = atan2(y − R, x) + π/2, and ψ_e = wrap(θ − ψ).

The law is δ = −K_E·e + K_HEAD·ψ_e (+ arctan(L·κ) with κ = 1/R when the feedforward is on), K_E = 0.2, K_HEAD = 1.0; start at the origin heading +x at 15 m/s, drag-balancing throttle every tick, no brake, 30 s.
The **steady-state error** is the mean of e (measured after each advance) over the last 10 s.
(Use `math.atan2(y, 1.0)` for arctan y, as the lesson does.)

1. Predict the steady-state error without the feedforward: in the steady state ψ_e ≈ 0, so the P term alone must supply the circle's wheel angle.
2. Measure it with and without the feedforward for R in (20, 40, 80).
3. Check the instrument: compute Unit 2's signed Menger curvature of three consecutive points 2 m apart (by arc length) on the circle of radius 40 starting at the origin, and compare with 1/40.

Define:

- `predicted_no_ff`: a dict from R to the predicted error (m, signed);
- `steady_no_ff`, `steady_ff`: dicts from R to the measured steady-state error (m, signed);
- `menger_40`: the Menger curvature of the three points (1/m, signed);
- `why_feedforward`: one sentence on what the feedforward supplies that the PD terms can only supply with an error, and why the prediction is slightly off at R = 20.

In [ ]:
RADII = (20.0, 40.0, 80.0)
predicted_no_ff = {}  # TODO
steady_no_ff = {}  # TODO
steady_ff = {}  # TODO
menger_40 = None  # TODO
why_feedforward = "TODO"

## Exercise 6 - Tune K_E on the track, beyond the noise

The reference uses K_E = 0.2.
Use `write_variant` to make submissions from `agents/unit-04.py` with `K_E` = 0.1, 0.2 and 0.4 (nothing else changed), and race each on `"oval"` with `repeat_races(..., n=5, car_factory=...)` on `KinematicCar(jitter=0.05)` (seeds 0–4).
For each race, the RMS offset is the root-mean-square of the telemetry's `offset` column over the rows with `held == 0`.

Define:

- `rms_medians`, `rms_ranges`: dicts from K_E to the median and the range (max − min) of the five RMS offsets (m);
- `time_medians`: a dict from K_E to the median race time (s);
- `all_clean`: `True` when all 15 races finished with 0 collisions;
- `best_k_e`: the K_E with the smallest RMS median;
- `beats_reference`: `True` when `best_k_e` differs from 0.2 **and** its RMS median is below the reference's by more than both ranges;
- `verdict`: one or two sentences: would you change the reference's K_E on this evidence alone, and what else would you want to measure first?

In [ ]:
GAINS = (0.1, 0.2, 0.4)
rms_medians = {}  # TODO
rms_ranges = {}  # TODO
time_medians = {}  # TODO
all_clean = None  # TODO
best_k_e = None  # TODO
beats_reference = None  # TODO
verdict = "TODO"

## Exercise 7 - Challenge: the derivative kick in the speed loop

Give the speed loop a D term and move its set point.
Use your `PID` with kp = 0.5, ki = 0.1, kd = 0.05 and `limits=(0.0, 1.0)`.
Drive `KinematicCar()` from rest with a target of 20 m/s; at t = 20 s (from tick 400 on, counting ticks from 0) the target becomes 25 m/s; run 40 s in all.
Race two versions:

- `"difference"`: `update(error, DT)`, the finite difference of the error;
- `"measured"`: `update(error, DT, rate=r)` with r the rate of the error **from the measured speed only**, r = −(v_n − v_{n−1}) / DT, where v_n is the speed at the start of tick n (r = 0 on the first tick).
  This is what the error's rate would be if the target never moved.

For each version record, every tick, the D term's contribution (kd times the rate used) and the command `update` returned.

Define:

- `kick`: a dict from the version name to the D term's contribution on tick 400 (signed);
- `peak_command`: a dict from the version name to the largest command over ticks 400 onwards;
- `second_step`: a dict from the version name to the `step_metrics` triple of the second step (the speeds from tick 400 onwards, from v0 = the speed at the start of tick 400 to 25 m/s);
- `kick_verdict`: one or two sentences: did the kick reach the car here, and why is the same kick worse in a steering loop (the lesson's §5)?

In [ ]:
kick = {}  # TODO
peak_command = {}  # TODO
second_step = {}  # TODO
kick_verdict = "TODO"

## Exercise 8 - Challenge: an I term on the steering instead of the feedforward

An I term also removes a steady error: could it replace the feedforward?
On Exercise 5's circle with R = 40 and no feedforward, add an integral term to the steering: δ = −K_E·e + K_HEAD·ψ_e + K_I·∫(−e)dt, using your `PID` on the error −e with `kd = K_HEAD` and `rate = ψ_e` (and no limits).
Run 60 s for K_I in (0.02, 0.05, 0.1).
For each, record e **after** each advance, as in Exercise 5, and report the steady-state error (mean e over the last 10 s) and the settling time to the band |e| ≤ 0.02 m (as in Exercise 4); do the same for the feedforward without an I term.

Then measure how long the corners of `test-circuit` last.
Take Unit 2's signed Menger curvature at every waypoint of `load_track("test-circuit")` (cyclic triples); a **corner** is a maximal run of consecutive waypoints with |κ| > 0.005 1/m, treating the waypoint list as **not** cyclic (a run does not continue from the last waypoint to the first).
Its duration at 20 m/s is its number of waypoints times the mean waypoint spacing, divided by 20; take the mean spacing as `track.length / len(track)` (the lap length over the number of waypoints).

Define:

- `steady_i`, `settle_i`: dicts from K_I to the steady-state error (m) and the settling time (s);
- `settle_ff`: the feedforward's settling time (s);
- `corner_seconds`: a sorted list of the corners' durations (s);
- `i_or_ff`: one or two sentences: which would you put in the submission, comparing the settling times with the corner durations.

In [ ]:
K_IS = (0.02, 0.05, 0.1)
steady_i = {}  # TODO
settle_i = {}  # TODO
settle_ff = None  # TODO
corner_seconds = []  # TODO
i_or_ff = "TODO"